# Urolithin A Target Search

**BioPipelines example** — finding candidate protein targets for a small molecule. RCSB is searched for structures whose co-crystallised ligands are chemically similar to urolithin A, Boltz2 co-folds the metabolite against each hit, and the candidates are ranked by predicted binding probability and interface confidence.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[colab]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba

In [ ]:
from biopipelines.pipeline import *
from biopipelines import Boltz2, PoseBusters

with Pipeline("Setup", "install", description="Install tools"):
    Boltz2.install()
    PoseBusters.install()

## Cell 3: Target Search Pipeline

Treats urolithin A — a gut-microbiome metabolite of ellagitannins — as a query and asks *which proteins might bind it*:

1. **RCSB** runs a chemical-similarity search (fingerprint similarity to the urolithin A SMILES), restricted to protein entities, and downloads the top 5 hits with their sequences.
2. **Boltz2** co-folds urolithin A against each candidate target and predicts interface confidence (ipTM) plus two affinity readouts.
3. **PoseBusters** validates each co-folded pose (geometry, clashes) — a high-confidence but physically broken pose shouldn't count as a hit.
4. **Panda** merges the search metadata, confidence, affinity and validation, keeps the geometry-valid complexes, and ranks the hits.
5. **Plot** visualises the candidates.

**Which affinity metric?** Boltz-2 reports two: `affinity_probability_binary` (0–1, "is this a binder vs a decoy?") and `affinity_pred_value` (= log10(IC50 in µM); lower = tighter). For a *target search* — *does this molecule bind this protein at all?* — `affinity_probability_binary` is the right ranking metric. `affinity_pred_value` (shown as `aff_uM = 10**value`) is meaningful only for ranking *among* genuine binders (hit-to-lead), so we use it as a secondary readout, not the primary rank.

A lightweight way to nominate structural hypotheses for a metabolite of interest before committing to assays.

In [ ]:
from biopipelines.pipeline import *
from biopipelines.rcsb import StructureAttribute, PolymerEntityType
from biopipelines import Boltz2, PoseBusters, Panda, Plot

UROA_SMILES = "Oc1ccc2c(c1)oc(=O)c1cc(O)ccc12"

Pipeline(project="UrolithinA",
         job="TargetSearchDemo",
         description="Colab demo: cofold UroA against 5 RCSB hits")

uroa = Ligand(smiles=UROA_SMILES, ids="UroA", codes="URO")

# Inverse screen: proteins whose co-crystal ligands are chemically similar to UroA.
candidates = RCSB(RCSB.Chemical(UROA_SMILES, match_type="fingerprint-similarity"),
                  StructureAttribute.PolymerMolecularFeatures.PolymerEntityType.equals(PolymerEntityType.PROTEIN),
                  max_results=5,
                  sort="score")

# Co-fold UroA against each candidate; validate the resulting poses.
cofolded = Boltz2(proteins=candidates, ligands=uroa)
busted   = PoseBusters(structures=cofolded, ligand=cofolded)   # reads UroA's code from the Boltz2 output

# Merge search metadata + confidence + affinity + validation; keep valid poses; rank by binder probability.
ranked = Panda(tables=[candidates.tables.search_results,
                       cofolded.tables.confidence,
                       cofolded.tables.affinity,
                       busted.tables.posebusters],
               operations=[Panda.merge(),
                           Panda.filter("all_pass == True"),
                           Panda.calculate({"aff_uM": "10**affinity_pred_value"}),
                           Panda.sort("affinity_probability_binary", ascending=False)])

Plot(Plot.Scatter(data=ranked.tables.result,
                  x="iptm",
                  y="affinity_probability_binary",
                  color="aff_uM",
                  title="UroA candidate targets — binder probability vs interface confidence",
                  xlabel="ipTM",
                  ylabel="Binding probability"),
    Plot.Bar(data=ranked.tables.result,
             x="pdb_id",
             y="affinity_probability_binary",
             title="UroA binding probability by PDB hit",
             xlabel="PDB hit",
             ylabel="Binding probability"))
ranked.tables.result